# Comparación de Estrategias Federadas vs Proactive Forest Base

## Objetivo
Comparar las métricas (Accuracy y Macro-F1) de cada estrategia federada (S1-S7 + PW) con los resultados base de Proactive Forest reportados por Nayma, usando **3 clientes** y promediando las métricas de los 3 clientes.

## Datasets
Car, Iris, Letter, Nursery, Optdigits, Sonar, Spambase, Vowel

## Configuración fija
- **n_clients = 3**
- **seed = 42**
- **n_estimators = 100**
- **distribution = iid**
- **alpha_pf = 0.45**
- **t_max = 100** (S2-S7)
- **local_weight = 0.5**
- **f1_weight = 0.6** (S4, S7, PW)
- **window_size = 7** (PW)
- **max_rounds = 15** (PW)
- **convergence_threshold = 0.002** (PW)

---
## 0. Imports y configuración común

In [1]:
import sys
from pathlib import Path

# Notebook está en src/interfaces/notebooks/ → buscar project root
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.infrastructure.dataset.dataset_factory import DatasetFactory

print(f'Project root: {ROOT}')
assert (ROOT / 'src').exists(), f'No se encontró src/ en {ROOT}'

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from src.application.orchestrators import FLEXOrchestrator
from src.domain.dataset.base_adapter import DatasetSplit

SEED = 42
N_CLIENTS = 3
N_ESTIMATORS = 100
ALPHA_PF = 0.45
T_MAX = 100
LOCAL_WEIGHT = 0.5
F1_WEIGHT = 0.6

np.random.seed(SEED)

Project root: C:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest


---
## 1. Dataset Loaders

In [2]:
# ============================================================================
# DATASET LOADING CONFIGURATION
# ============================================================================
# Using DatasetFactory to standardize all dataset adapters.

def load_dataset(name):
    """Standardized loader using DatasetFactory."""
    return DatasetFactory.load_from_config({"type": name.capitalize()}, project_root=ROOT)

DATASETS = {
    'car': lambda: load_dataset('car'),
    'iris': lambda: load_dataset('iris'),
    'letter': lambda: load_dataset('letter'),
    'nursery': lambda: load_dataset('nursery'),
    'optdigits': lambda: load_dataset('optdigits'),
    'sonar': lambda: load_dataset('sonar'),
    'spambase': lambda: load_dataset('spambase'),
    'vowel': lambda: load_dataset('vowel'),
}


---
## 2. Función de experimento genérica

In [3]:
def build_config(strategy, n_clients=N_CLIENTS, n_estimators=N_ESTIMATORS, seed=SEED):
    """Construye config con los parámetros unificados para cualquier estrategia."""
    cfg = {
        'federation': {'n_clients': n_clients, 'distribution': 'iid', 'seed': seed},
        'model': {
            'n_estimators': n_estimators, 'alpha': ALPHA_PF,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {'strategy': strategy, 't_max': T_MAX},
        'prediction': {'local_weight': LOCAL_WEIGHT, 'global_weight': 1.0 - LOCAL_WEIGHT},
        'verbose': False, 'seed': seed,
    }

    # S4, S7, PW → f1_weight
    if strategy in ('S4', 'S7', 'PW'):
        cfg['aggregation']['f1_weight'] = F1_WEIGHT
        cfg['aggregation']['pcd_weight'] = 1.0 - F1_WEIGHT

    # PW → parámetros exclusivos
    if strategy == 'PW':
        cfg['aggregation']['window_size'] = 7
        cfg['aggregation']['max_rounds'] = 15
        cfg['aggregation']['convergence_threshold'] = 0.002

    return cfg


def run_experiment(dataset_name, strategy):
    """Ejecuta una combinación dataset × estrategia y devuelve dict con métricas por cliente."""
    ds = DATASETS[dataset_name]()
    cfg = build_config(strategy)
    np.random.seed(SEED)

    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    results = orch.run_federated_round()

    # Métricas por cliente (híbridas)
    client_metrics = {}
    for cid, preds in results.client_hybrid_predictions.items():
        acc = accuracy_score(results.y_test, preds)
        f1 = f1_score(results.y_test, preds, average='macro', zero_division=0)
        client_metrics[cid] = {'accuracy': acc, 'macro_f1': f1}

    return {
        'dataset': dataset_name,
        'strategy': strategy,
        'global_accuracy': results.global_accuracy,
        'global_macro_f1': results.global_macro_f1,
        'client_metrics': client_metrics,
        'n_trees_global': results.n_trees_global,
    }

---
## 3. Ejecutar cada estrategia (celda independiente por si acaso)

> **Nota:** Cada celda ejecuta la estrategia para **todos los datasets**. Los resultados se acumulan en `all_results`.

In [4]:
# Celda compartida para almacenar resultados
all_results = {}  # {(dataset, strategy): result_dict}

In [5]:
# ── S1: Simple Pool ──────────────────────────────────────────────────────
STRATEGY = 'S1'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S1
  ▶ car... F1=0.0000
  ▶ iris... F1=0.0000
  ▶ letter... 

KeyboardInterrupt: 

In [ ]:
# ── S2: Global Accuracy + PF ─────────────────────────────────────────────
STRATEGY = 'S2'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S3: Global Macro-F1 + PF ─────────────────────────────────────────────
STRATEGY = 'S3'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S4: Global F1 + PCD + PF ─────────────────────────────────────────────
STRATEGY = 'S4'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S5: Per-Client Accuracy + PF ─────────────────────────────────────────
STRATEGY = 'S5'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S6: Per-Client Macro-F1 + PF ─────────────────────────────────────────
STRATEGY = 'S6'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S7: Per-Client F1 + PCD + PF ─────────────────────────────────────────
STRATEGY = 'S7'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── PW: Progressive Windows ──────────────────────────────────────────────
STRATEGY = 'PW'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

---
## 4. Tabla comparativa final

Se calcula el **promedio de Accuracy y F1 de los 3 clientes** para cada combinación dataset × estrategia.

In [ ]:
# ── Resultados base de Nayma (Proactive Forest) ──────────────────────────
NAYMA_RESULTS = {
    'car':       {'accuracy': 0.976625780, 'f1': 0.945782},
    'iris':      {'accuracy': 0.956000000, 'f1': 0.954981},
    'letter':    {'accuracy': 0.965045493, 'f1': 0.965253},
    'nursery':   {'accuracy': 0.995910870, 'f1': 0.954848},
    'optdigits': {'accuracy': 0.983235639, 'f1': 0.982218},
    'sonar':     {'accuracy': 0.848298701, 'f1': 0.823483},
    'spambase':  {'accuracy': 0.953879759, 'f1': 0.952755},
    'vowel':     {'accuracy': 0.971919192, 'f1': 0.968468},
}

STRATEGIES = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'PW']
DATASET_ORDER = ['car', 'iris', 'letter', 'nursery', 'optdigits', 'sonar', 'spambase', 'vowel']

In [ ]:
# ── Construir tabla comparativa ──────────────────────────────────────────
rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}

    # Columna PF (Nayma)
    row['Accuracy_PF'] = round(NAYMA_RESULTS[ds]['accuracy'], 6)
    row['F1_PF'] = round(NAYMA_RESULTS[ds]['f1'], 6)

    # Columnas por estrategia
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            # Promedio de los 3 clientes
            client_accs = [m['accuracy'] for m in res['client_metrics'].values()]
            client_f1s = [m['macro_f1'] for m in res['client_metrics'].values()]
            row[f'Accuracy_{strat}'] = round(np.mean(client_accs), 6)
            row[f'F1_{strat}'] = round(np.mean(client_f1s), 6)
        else:
            row[f'Accuracy_{strat}'] = None
            row[f'F1_{strat}'] = None

    rows.append(row)

df_comparison = pd.DataFrame(rows)

# Reordenar columnas: BD, PF, luego cada estrategia
col_order = ['BD', 'Accuracy_PF', 'F1_PF']
for strat in STRATEGIES:
    col_order += [f'Accuracy_{strat}', f'F1_{strat}']
df_comparison = df_comparison[col_order]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.6f}' if pd.notna(x) else '')

print(df_comparison.to_string(index=False))

In [ ]:
# ── Guardar tabla como CSV y Excel ───────────────────────────────────────
out_dir = ROOT / 'results' / 'comparison_vs_nayma'
out_dir.mkdir(parents=True, exist_ok=True)

df_comparison.to_csv(out_dir / 'comparison_table.csv', index=False)
df_comparison.to_excel(out_dir / 'comparison_table.xlsx', index=False)
print(f"\n✅ Tabla guardada en: {out_dir}")

In [ ]:
# ── Tabla de diferencias vs Nayma (estrategia - PF) ─────────────────────
diff_rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            client_accs = [m['accuracy'] for m in res['client_metrics'].values()]
            client_f1s = [m['macro_f1'] for m in res['client_metrics'].values()]
            avg_acc = np.mean(client_accs)
            avg_f1 = np.mean(client_f1s)
            row[f'ΔAcc_{strat}'] = round(avg_acc - NAYMA_RESULTS[ds]['accuracy'], 6)
            row[f'ΔF1_{strat}'] = round(avg_f1 - NAYMA_RESULTS[ds]['f1'], 6)
        else:
            row[f'ΔAcc_{strat}'] = None
            row[f'ΔF1_{strat}'] = None
    diff_rows.append(row)

diff_cols = ['BD']
for strat in STRATEGIES:
    diff_cols += [f'ΔAcc_{strat}', f'ΔF1_{strat}']

df_diff = pd.DataFrame(diff_rows)[diff_cols]
print(df_diff.to_string(index=False))

df_diff.to_csv(out_dir / 'comparison_diff_vs_PF.csv', index=False)
print(f"\n✅ Diferencias guardadas en: {out_dir / 'comparison_diff_vs_PF.csv'}")